<a href="https://colab.research.google.com/github/nehansa2003/NLP_project/blob/main/Game_Review_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
!pip install -q yake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 16.1 MB/s eta 0:00:00


In [39]:
import pandas as pd
import numpy as np
import torch
import json
from torch.utils.data import DataLoader
from transformers import (AutoTokenizer,AutoModelForSequenceClassification)
from sklearn.feature_extraction.text import TfidfVectorizer
import yake
import re
from collections import Counter
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_PATH = '/content/drive/MyDrive/Game_Review_NLP'

data = pd.read_csv(
    f'{PROJECT_PATH}/all_reviews_processed_v3.csv'
)

print("Dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

Dataset shape: (100027, 19)

Columns:
['game_id', 'game_name', 'game_url', 'review_id', 'author', 'outlet', 'score', 'date', 'review_text', 'review_url', 'clean_review_text', 'word_count', 'score_normalized', 'sentiment', 'score_normalized_v2', 'sentiment_v2', 'score_normalized_v3', 'sentiment_v3', 'sentiment_final']


In [4]:
required_columns = [
    'game_id',
    'game_name',
    'review_text'
]

for col in required_columns:
    print(
        col,
        "✓" if col in data.columns else "✗ MISSING"
    )

game_id ✓
game_name ✓
review_text ✓


In [5]:
print(
    data[
        [
            'game_id',
            'game_name',
            'review_text'
        ]
    ].head()
)

   game_id            game_name  \
0        1  Super Mario Odyssey   
1        1  Super Mario Odyssey   
2        1  Super Mario Odyssey   
3        1  Super Mario Odyssey   
4        1  Super Mario Odyssey   

                                         review_text  
0  Super Mario Odyssey is a spectacular return to...  
1  Mario's games have been around for almost as l...  
2  One of the most daring and influential game de...  
3  Super Mario Odyssey successfully brings the se...  
4  With hundreds of moons to collect and a dizzyi...  


Remove reviews without text

In [6]:
nlp_data = data[
    data['review_text'].notna()
].copy()

nlp_data['review_text'] = (
    nlp_data['review_text']
    .astype(str)
    .str.strip()
)

nlp_data = nlp_data[
    nlp_data['review_text'].str.len() > 0
].copy()

print(
    "Reviews available for NLP:",
    len(nlp_data)
)

Reviews available for NLP: 100027


Load your final DistilBERT model

In [7]:
FINAL_MODEL_DIR = (
    f'{PROJECT_PATH}/final_distilbert_game_sentiment'
)

tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

model = AutoModelForSequenceClassification.from_pretrained(
    FINAL_MODEL_DIR
)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model.to(device)
model.eval()

print("Device:", device)
print("Final DistilBERT loaded")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Device: cuda
Final DistilBERT loaded


Load the frozen threshold

In [8]:
with open(
    f'{FINAL_MODEL_DIR}/threshold_config.json',
    'r'
) as f:

    threshold_config = json.load(f)

BEST_NEGATIVE_THRESHOLD = (
    threshold_config['negative_threshold']
)

print(
    "Frozen negative threshold:",
    BEST_NEGATIVE_THRESHOLD
)

Frozen negative threshold: 0.3


Predict sentiment for reviews

In [9]:
texts = nlp_data[
    'review_text'
].tolist()

batch_size = 16

all_probabilities = []

for start in range(
    0,
    len(texts),
    batch_size
):

    batch_texts = texts[
        start:start + batch_size
    ]

    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():

        outputs = model(
            **encoded
        )

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )

    all_probabilities.append(
        probabilities.cpu().numpy()
    )

    if (
        start // batch_size
    ) % 100 == 0:

        print(
            f"Processed "
            f"{min(start + batch_size, len(texts)):,}"
            f" / {len(texts):,}"
        )

all_probabilities = np.vstack(
    all_probabilities
)

print(
    "Probability matrix:",
    all_probabilities.shape
)

Processed 16 / 100,027
Processed 1,616 / 100,027
Processed 3,216 / 100,027
Processed 4,816 / 100,027
Processed 6,416 / 100,027
Processed 8,016 / 100,027
Processed 9,616 / 100,027
Processed 11,216 / 100,027
Processed 12,816 / 100,027
Processed 14,416 / 100,027
Processed 16,016 / 100,027
Processed 17,616 / 100,027
Processed 19,216 / 100,027
Processed 20,816 / 100,027
Processed 22,416 / 100,027
Processed 24,016 / 100,027
Processed 25,616 / 100,027
Processed 27,216 / 100,027
Processed 28,816 / 100,027
Processed 30,416 / 100,027
Processed 32,016 / 100,027
Processed 33,616 / 100,027
Processed 35,216 / 100,027
Processed 36,816 / 100,027
Processed 38,416 / 100,027
Processed 40,016 / 100,027
Processed 41,616 / 100,027
Processed 43,216 / 100,027
Processed 44,816 / 100,027
Processed 46,416 / 100,027
Processed 48,016 / 100,027
Processed 49,616 / 100,027
Processed 51,216 / 100,027
Processed 52,816 / 100,027
Processed 54,416 / 100,027
Processed 56,016 / 100,027
Processed 57,616 / 100,027
Processed 5

Apply the 0.30 threshold

In [10]:
predicted_ids = np.argmax(
    all_probabilities,
    axis=1
)

negative_mask = (
    all_probabilities[:, 0]
    >= BEST_NEGATIVE_THRESHOLD
)

predicted_ids[
    negative_mask
] = 0

In [11]:
id2label = {
    0: 'negative',
    1: 'neutral',
    2: 'positive'
}

predicted_sentiment = [
    id2label[int(x)]
    for x in predicted_ids
]

In [12]:
nlp_data[
    'predicted_sentiment'
] = predicted_sentiment

nlp_data[
    'negative_probability'
] = all_probabilities[:, 0]

nlp_data[
    'neutral_probability'
] = all_probabilities[:, 1]

nlp_data[
    'positive_probability'
] = all_probabilities[:, 2]

In [13]:
print(
    nlp_data[
        'predicted_sentiment'
    ].value_counts()
)

predicted_sentiment
positive    85350
neutral     14362
negative      315
Name: count, dtype: int64


Save the review-level predictions

In [14]:
PREDICTIONS_PATH = (
    f'{PROJECT_PATH}/all_review_predictions.csv'
)

nlp_data.to_csv(
    PREDICTIONS_PATH,
    index=False
)

print(
    "✓ Predictions saved:"
)

print(PREDICTIONS_PATH)

✓ Predictions saved:
/content/drive/MyDrive/Game_Review_NLP/all_review_predictions.csv


Calculate game-level sentiment

In [15]:
game_counts = (
    nlp_data
    .groupby(
        [
            'game_id',
            'game_name',
            'predicted_sentiment'
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

In [16]:
game_counts = game_counts.reindex(
    columns=[
        'negative',
        'neutral',
        'positive'
    ],
    fill_value=0
)

Convert to percentages

In [17]:
game_percentages = (
    game_counts
    .div(
        game_counts.sum(axis=1),
        axis=0
    )
    .mul(100)
    .round(2)
)

game_percentages.columns = [
    'negative_pct',
    'neutral_pct',
    'positive_pct'
]

display(
    game_percentages.head()
)

,,negative_pct,neutral_pct,positive_pct
game_id,game_name,,,
1,Super Mario Odyssey,0.0,1.97,98.03
2,Baldur's Gate 3,0.0,3.11,96.89
3,The Legend of Zelda: Breath of the Wild,0.0,2.47,97.53
4,Red Dead Redemption 2,0.0,4.17,95.83
5,The Legend of Zelda: Tears of the Kingdom,0.0,4.94,95.06


Add review count

In [18]:
review_counts = (
    nlp_data
    .groupby(
        [
            'game_id',
            'game_name'
        ]
    )
    .size()
    .rename(
        'review_count'
    )
)

In [19]:
game_intelligence = (
    review_counts
    .to_frame()
    .join(game_percentages)
    .reset_index()
)

Determine overall sentiment

In [20]:
def get_overall_sentiment(row):

    sentiment_values = {
        'positive': row['positive_pct'],
        'neutral': row['neutral_pct'],
        'negative': row['negative_pct']
    }

    return max(
        sentiment_values,
        key=sentiment_values.get
    )


game_intelligence[
    'overall_sentiment'
] = game_intelligence.apply(
    get_overall_sentiment,
    axis=1
)

Sort games by positive sentiment

In [21]:
most_positive_games = (
    game_intelligence
    .sort_values(
        'positive_pct',
        ascending=False
    )
)

display(
    most_positive_games.head(20)
)

,game_id,game_name,review_count,negative_pct,neutral_pct,positive_pct,overall_sentiment
65,66,Satisfactory,26,0.0,0.0,100.0,positive
1999,2000,Stranger of Sword City Revisited,3,0.0,0.0,100.0,positive
20,21,Dead Cells: The Queen & the Sea,4,0.0,0.0,100.0,positive
481,482,FAITH: The Unholy Trinity,11,0.0,0.0,100.0,positive
38,39,Super Smash Bros. Ultimate,146,0.0,0.0,100.0,positive
482,483,Frog Detective 3: Corruption at Cowboy County,5,0.0,0.0,100.0,positive
487,488,Tombwater,11,0.0,0.0,100.0,positive
472,473,Echo Arena,3,0.0,0.0,100.0,positive
438,439,Later Alligator,7,0.0,0.0,100.0,positive
471,472,Polybius,7,0.0,0.0,100.0,positive


In [22]:
most_negative_games = (
    game_intelligence
    .sort_values(
        'negative_pct',
        ascending=False
    )
)

display(
    most_negative_games.head(20)
)

,game_id,game_name,review_count,negative_pct,neutral_pct,positive_pct,overall_sentiment
652,653,Kingdom Hearts: All-In-One Package,4,25.00,0.00,75.00,positive
1266,1267,The Banished Vault,5,20.00,20.00,60.00,positive
1231,1232,Max Payne 3,6,16.67,0.00,83.33,positive
636,637,Clannad,6,16.67,16.67,66.67,positive
1406,1407,The Last of Us Part I,174,12.07,14.37,73.56,positive
1261,1262,Umurangi Generation,17,11.76,11.76,76.47,positive
1881,1882,Invincible Presents: Atom Eve,10,10.00,40.00,50.00,positive
382,383,Far From Noise,10,10.00,20.00,70.00,positive
1670,1671,Demon's Mirror,10,10.00,20.00,70.00,positive
1188,1189,Retro City Rampage DX,11,9.09,27.27,63.64,positive


Save game-level intelligence

In [23]:
GAME_INTELLIGENCE_PATH = (
    f'{PROJECT_PATH}/game_intelligence.csv'
)

game_intelligence.to_csv(
    GAME_INTELLIGENCE_PATH,
    index=False
)

print(
    "✓ Game intelligence saved:"
)

print(
    GAME_INTELLIGENCE_PATH
)

✓ Game intelligence saved:
/content/drive/MyDrive/Game_Review_NLP/game_intelligence.csv


Test the original idea

In [24]:
GAME_NAME = 'Super Mario Odyssey'

selected_game = game_intelligence[
    game_intelligence['game_name']
    == GAME_NAME
]

display(
    selected_game
)

,game_id,game_name,review_count,negative_pct,neutral_pct,positive_pct,overall_sentiment
0,1,Super Mario Odyssey,152,0.0,1.97,98.03,positive


Get all reviews for the selected game

In [25]:
game_reviews = nlp_data[
    nlp_data['game_name']
    == GAME_NAME
].copy()

print(
    "Reviews:",
    len(game_reviews)
)

display(
    game_reviews[
        [
            'game_name',
            'predicted_sentiment',
            'review_text'
        ]
    ].head(10)
)

Reviews: 152


,game_name,predicted_sentiment,review_text
0,Super Mario Odyssey,positive,Super Mario Odyssey is a spectacular return to...
1,Super Mario Odyssey,positive,Mario's games have been around for almost as l...
2,Super Mario Odyssey,positive,One of the most daring and influential game de...
3,Super Mario Odyssey,positive,Super Mario Odyssey successfully brings the se...
4,Super Mario Odyssey,positive,With hundreds of moons to collect and a dizzyi...
5,Super Mario Odyssey,positive,"For a character nearing 40 years old, it's ama..."
6,Super Mario Odyssey,positive,Super Mario Odyssey is a massive and magnifice...
7,Super Mario Odyssey,positive,The soul of Super Mario 64 is alive and well i...
8,Super Mario Odyssey,positive,Every element gelled so well that I was simply...
9,Super Mario Odyssey,positive,"Odyssey is the best Mario game in many, many y..."


# Theme Intelligence

In [27]:
predictions = pd.read_csv(
    f'{PROJECT_PATH}/all_review_predictions.csv'
)

print("Shape:", predictions.shape)
print(predictions.columns.tolist())

Shape: (100027, 23)
['game_id', 'game_name', 'game_url', 'review_id', 'author', 'outlet', 'score', 'date', 'review_text', 'review_url', 'clean_review_text', 'word_count', 'score_normalized', 'sentiment', 'score_normalized_v2', 'sentiment_v2', 'score_normalized_v3', 'sentiment_v3', 'sentiment_final', 'predicted_sentiment', 'negative_probability', 'neutral_probability', 'positive_probability']


In [28]:
GAME_NAME = 'Super Mario Odyssey'

game_reviews = predictions[
    predictions['game_name'] == GAME_NAME
].copy()

print(
    f"Game: {GAME_NAME}"
)

print(
    f"Reviews: {len(game_reviews):,}"
)

Game: Super Mario Odyssey
Reviews: 152


In [29]:
print(
    predictions['game_name']
    .drop_duplicates()
    .head(30)
    .tolist()
)

['Super Mario Odyssey', "Baldur's Gate 3", 'The Legend of Zelda: Breath of the Wild', 'Red Dead Redemption 2', 'The Legend of Zelda: Tears of the Kingdom', 'Grand Theft Auto IV', 'Elden Ring', 'Astro Bot', 'The Legend of Zelda: Tears of the Kingdom Nintendo Switch 2 Edition', 'The Last of Us Remastered', 'God of War', 'Hades II', 'Persona 5 Royal', 'Metroid Prime Remastered', 'Persona 5', 'Hades', 'Elden Ring: Shadow of the Erdtree', 'Divinity: Original Sin 2', 'Journey', 'The House in Fata Morgana', 'Dead Cells: The Queen & the Sea', 'Undertale', 'Super Mario 3D World', "Uncharted 4: A Thief's End", 'Metaphor: ReFantazio', 'The Witcher 3: Wild Hunt', 'Final Fantasy XIV: Endwalker', 'The Last of Us Part II', 'The Witcher 3: Wild Hunt - Blood and Wine', 'Clair Obscur: Expedition 33']


In [30]:
predictions['review_text'] = (
    predictions['review_text']
    .fillna('')
    .astype(str)
)

predictions['game_name'] = (
    predictions['game_name']
    .fillna('Unknown Game')
    .astype(str)
)

predictions['predicted_sentiment'] = (
    predictions['predicted_sentiment']
    .str.lower()
    .str.strip()
)

predictions = predictions[
    predictions['review_text'].str.len() > 10
].copy()

print(
    "Reviews available for NLP:",
    len(predictions)
)

Reviews available for NLP: 100027


Define Aspects

In [31]:
ASPECTS = {

    'Gameplay': [
        'gameplay',
        'mechanics',
        'combat',
        'controls',
        'movement',
        'platforming',
        'exploration'
    ],

    'Story': [
        'story',
        'narrative',
        'plot',
        'writing',
        'dialogue'
    ],

    'Graphics': [
        'graphics',
        'visuals',
        'visual',
        'art style',
        'animation'
    ],

    'Characters': [
        'character',
        'characters',
        'protagonist',
        'villain'
    ],

    'Level Design': [
        'level design',
        'level',
        'levels',
        'world design',
        'environment',
        'map'
    ],

    'Music & Sound': [
        'music',
        'soundtrack',
        'sound',
        'audio'
    ],

    'Difficulty': [
        'difficulty',
        'difficult',
        'easy',
        'hard',
        'challenging'
    ],

    'Performance': [
        'performance',
        'fps',
        'frame rate',
        'loading',
        'bug',
        'bugs',
        'technical'
    ],

    'Replayability': [
        'replayability',
        'replay',
        'replay value',
        'content',
        'length'
    ]
}

print(
    "Number of aspects:",
    len(ASPECTS)
)

Number of aspects: 9


analysis function

In [32]:
def analyze_aspects(
    game_name,
    data
):

    game_data = data[
        data['game_name'] == game_name
    ].copy()

    results = []

    for aspect, keywords in ASPECTS.items():

        # Create regex safely
        escaped_keywords = [
            keyword.replace(
                ' ',
                r'\s+'
            )
            for keyword in keywords
        ]

        pattern = (
            r'\b(?:'
            + '|'.join(escaped_keywords)
            + r')\b'
        )

        mask = (
            game_data['review_text']
            .str.lower()
            .str.contains(
                pattern,
                regex=True,
                na=False
            )
        )

        aspect_reviews = game_data[mask]

        if len(aspect_reviews) == 0:
            continue

        sentiment_counts = (
            aspect_reviews[
                'predicted_sentiment'
            ]
            .value_counts()
        )

        mentions = len(
            aspect_reviews
        )

        positive = sentiment_counts.get(
            'positive',
            0
        )

        neutral = sentiment_counts.get(
            'neutral',
            0
        )

        negative = sentiment_counts.get(
            'negative',
            0
        )

        results.append({

            'game_name': game_name,

            'aspect': aspect,

            'mentions': mentions,

            'positive_count': positive,

            'neutral_count': neutral,

            'negative_count': negative,

            'positive_pct': round(
                positive / mentions * 100,
                2
            ),

            'neutral_pct': round(
                neutral / mentions * 100,
                2
            ),

            'negative_pct': round(
                negative / mentions * 100,
                2
            )
        })

    return pd.DataFrame(results)

In [33]:
test_game = (
    predictions['game_name']
    .value_counts()
    .index[0]
)

print(
    "Testing game:",
    test_game
)

test_aspects = analyze_aspects(
    test_game,
    predictions
)

display(test_aspects)

Testing game: God of War


,game_name,aspect,mentions,positive_count,neutral_count,negative_count,positive_pct,neutral_pct,negative_pct
0,God of War,Gameplay,63,62,1,0,98.41,1.59,0.0
1,God of War,Story,72,69,3,0,95.83,4.17,0.0
2,God of War,Graphics,24,24,0,0,100.00,0.00,0.0
3,God of War,Characters,22,22,0,0,100.00,0.00,0.0
4,God of War,Level Design,9,9,0,0,100.00,0.00,0.0
5,God of War,Music & Sound,12,12,0,0,100.00,0.00,0.0
6,God of War,Difficulty,7,7,0,0,100.00,0.00,0.0
7,God of War,Performance,13,12,1,0,92.31,7.69,0.0
8,God of War,Replayability,8,8,0,0,100.00,0.00,0.0


Run for all games

In [34]:
all_aspect_results = []

games = sorted(
    predictions['game_name']
    .dropna()
    .unique()
)

print(
    "Analyzing",
    len(games),
    "games..."
)

for i, game_name in enumerate(games):

    result = analyze_aspects(
        game_name,
        predictions
    )

    if not result.empty:

        all_aspect_results.append(
            result
        )

    if (i + 1) % 100 == 0:

        print(
            f"Processed {i + 1}/{len(games)} games"
        )

aspect_analysis = pd.concat(
    all_aspect_results,
    ignore_index=True
)

print(
    "\n✓ Layer 5 complete"
)

print(
    "Rows:",
    len(aspect_analysis)
)

Analyzing 2000 games...
Processed 100/2000 games
Processed 200/2000 games
Processed 300/2000 games
Processed 400/2000 games
Processed 500/2000 games
Processed 600/2000 games
Processed 700/2000 games
Processed 800/2000 games
Processed 900/2000 games
Processed 1000/2000 games
Processed 1100/2000 games
Processed 1200/2000 games
Processed 1300/2000 games
Processed 1400/2000 games
Processed 1500/2000 games
Processed 1600/2000 games
Processed 1700/2000 games
Processed 1800/2000 games
Processed 1900/2000 games
Processed 2000/2000 games

✓ Layer 5 complete
Rows: 12941


In [35]:
ASPECT_FILE = (
    f'{PROJECT_PATH}/aspect_analysis.csv'
)

aspect_analysis.to_csv(
    ASPECT_FILE,
    index=False
)

print(
    "✓ Saved:",
    ASPECT_FILE
)

✓ Saved: /content/drive/MyDrive/Game_Review_NLP/aspect_analysis.csv


In [36]:
display(
    aspect_analysis.head(20)
)

,game_name,aspect,mentions,positive_count,neutral_count,negative_count,positive_pct,neutral_pct,negative_pct
0,007 First Light,Gameplay,44,43,1,0,97.73,2.27,0.0
1,007 First Light,Story,44,43,1,0,97.73,2.27,0.0
2,007 First Light,Graphics,6,6,0,0,100.00,0.00,0.0
3,007 First Light,Characters,28,26,2,0,92.86,7.14,0.0
4,007 First Light,Level Design,9,9,0,0,100.00,0.00,0.0
5,007 First Light,Music & Sound,2,1,0,1,50.00,0.00,50.0
6,007 First Light,Difficulty,5,5,0,0,100.00,0.00,0.0
7,007 First Light,Performance,11,11,0,0,100.00,0.00,0.0
8,007 First Light,Replayability,4,2,2,0,50.00,50.00,0.0
9,1000xRESIST,Gameplay,8,7,1,0,87.50,12.50,0.0


In [37]:
display(
    aspect_analysis[
        aspect_analysis['game_name'] == test_game
    ]
    .sort_values(
        'mentions',
        ascending=False
    )
)

,game_name,aspect,mentions,positive_count,neutral_count,negative_count,positive_pct,neutral_pct,negative_pct
4389,God of War,Story,72,69,3,0,95.83,4.17,0.0
4388,God of War,Gameplay,63,62,1,0,98.41,1.59,0.0
4390,God of War,Graphics,24,24,0,0,100.00,0.00,0.0
4391,God of War,Characters,22,22,0,0,100.00,0.00,0.0
4395,God of War,Performance,13,12,1,0,92.31,7.69,0.0
4393,God of War,Music & Sound,12,12,0,0,100.00,0.00,0.0
4392,God of War,Level Design,9,9,0,0,100.00,0.00,0.0
4396,God of War,Replayability,8,8,0,0,100.00,0.00,0.0
4394,God of War,Difficulty,7,7,0,0,100.00,0.00,0.0


Automatic Key Phrases

Key phrase extraction function

In [40]:
def extract_key_phrases(
    game_name,
    data,
    sentiment,
    max_phrases=15
):

    reviews = data[
        (data['game_name'] == game_name)
        &
        (
            data['predicted_sentiment']
            == sentiment
        )
    ]['review_text']

    reviews = reviews.dropna()

    if len(reviews) == 0:
        return pd.DataFrame(
            columns=[
                'game_name',
                'sentiment',
                'phrase',
                'score'
            ]
        )

    # Combine reviews
    text = ' '.join(
        reviews.tolist()
    )

    # Limit enormous text
    text = text[:500000]

    extractor = yake.KeywordExtractor(
        lan='en',
        n=3,
        dedupLim=0.7,
        top=max_phrases
    )

    keywords = extractor.extract_keywords(
        text
    )

    results = []

    for phrase, score in keywords:

        phrase = phrase.strip()

        if len(phrase) < 3:
            continue

        results.append({

            'game_name': game_name,

            'sentiment': sentiment,

            'phrase': phrase,

            'score': score
        })

    return pd.DataFrame(results)

In [41]:
positive_phrases = extract_key_phrases(
    test_game,
    predictions,
    'positive'
)

negative_phrases = extract_key_phrases(
    test_game,
    predictions,
    'negative'
)

print("POSITIVE PHRASES")

display(
    positive_phrases
)

print("\nNEGATIVE PHRASES")

display(
    negative_phrases
)

POSITIVE PHRASES


,game_name,sentiment,phrase,score
0,God of War,positive,God of War,0.000006
1,God of War,positive,War,0.000113
2,God of War,positive,God,0.000162
3,God of War,positive,Santa Monica Studio,0.000286
4,God of War,positive,Santa Monica,0.000579
5,God of War,positive,Sony Santa Monica,0.000692
6,God of War,positive,War game,0.000724
7,God of War,positive,game,0.000878
8,God of War,positive,War series,0.001068
9,God of War,positive,Santa Monica Studios,0.001286



NEGATIVE PHRASES


,game_name,sentiment,phrase,score


In [42]:
all_phrase_results = []

for i, game_name in enumerate(games):

    for sentiment in [
        'positive',
        'neutral',
        'negative'
    ]:

        result = extract_key_phrases(
            game_name,
            predictions,
            sentiment,
            max_phrases=15
        )

        if not result.empty:

            all_phrase_results.append(
                result
            )

    if (i + 1) % 100 == 0:

        print(
            f"Processed {i + 1}/{len(games)} games"
        )

key_phrases = pd.concat(
    all_phrase_results,
    ignore_index=True
)

print(
    "✓ Layer 6 complete"
)

print(
    "Rows:",
    len(key_phrases)
)

Processed 100/2000 games
Processed 200/2000 games
Processed 300/2000 games
Processed 400/2000 games
Processed 500/2000 games
Processed 600/2000 games
Processed 700/2000 games
Processed 800/2000 games
Processed 900/2000 games
Processed 1000/2000 games
Processed 1100/2000 games
Processed 1200/2000 games
Processed 1300/2000 games
Processed 1400/2000 games
Processed 1500/2000 games
Processed 1600/2000 games
Processed 1700/2000 games
Processed 1800/2000 games
Processed 1900/2000 games
Processed 2000/2000 games
✓ Layer 6 complete
Rows: 60544


In [43]:
PHRASE_FILE = (
    f'{PROJECT_PATH}/key_phrases.csv'
)

key_phrases.to_csv(
    PHRASE_FILE,
    index=False
)

print(
    "✓ Saved:",
    PHRASE_FILE
)

✓ Saved: /content/drive/MyDrive/Game_Review_NLP/key_phrases.csv


Generate advantages/disadvantages

In [44]:
def generate_pros_cons(
    game_name,
    aspect_data,
    phrase_data
):

    game_aspects = aspect_data[
        aspect_data['game_name']
        == game_name
    ].copy()

    game_phrases = phrase_data[
        phrase_data['game_name']
        == game_name
    ].copy()

    # Minimum mentions
    game_aspects = game_aspects[
        game_aspects['mentions'] >= 3
    ]

    advantages = game_aspects[
        game_aspects['positive_pct']
        >= 60
    ].sort_values(
        'positive_pct',
        ascending=False
    )

    disadvantages = game_aspects[
        game_aspects['negative_pct']
        >= 20
    ].sort_values(
        'negative_pct',
        ascending=False
    )

    positive_phrases = game_phrases[
        game_phrases['sentiment']
        == 'positive'
    ]

    negative_phrases = game_phrases[
        game_phrases['sentiment']
        == 'negative'
    ]

    return {
        'advantages': advantages,
        'disadvantages': disadvantages,
        'positive_phrases': positive_phrases,
        'negative_phrases': negative_phrases
    }

In [45]:
pros_cons = generate_pros_cons(
    test_game,
    aspect_analysis,
    key_phrases
)

print("ADVANTAGES")

display(
    pros_cons['advantages']
)

print("\nDISADVANTAGES")

display(
    pros_cons['disadvantages']
)

print("\nPOSITIVE PHRASES")

display(
    pros_cons['positive_phrases']
)

print("\nNEGATIVE PHRASES")

display(
    pros_cons['negative_phrases']
)

ADVANTAGES


,game_name,aspect,mentions,positive_count,neutral_count,negative_count,positive_pct,neutral_pct,negative_pct
4390,God of War,Graphics,24,24,0,0,100.00,0.00,0.0
4396,God of War,Replayability,8,8,0,0,100.00,0.00,0.0
4391,God of War,Characters,22,22,0,0,100.00,0.00,0.0
4392,God of War,Level Design,9,9,0,0,100.00,0.00,0.0
4393,God of War,Music & Sound,12,12,0,0,100.00,0.00,0.0
4394,God of War,Difficulty,7,7,0,0,100.00,0.00,0.0
4388,God of War,Gameplay,63,62,1,0,98.41,1.59,0.0
4389,God of War,Story,72,69,3,0,95.83,4.17,0.0
4395,God of War,Performance,13,12,1,0,92.31,7.69,0.0



DISADVANTAGES


,game_name,aspect,mentions,positive_count,neutral_count,negative_count,positive_pct,neutral_pct,negative_pct



POSITIVE PHRASES


,game_name,sentiment,phrase,score
20139,God of War,positive,God of War,0.000006
20140,God of War,positive,War,0.000113
20141,God of War,positive,God,0.000162
20142,God of War,positive,Santa Monica Studio,0.000286
20143,God of War,positive,Santa Monica,0.000579
20144,God of War,positive,Sony Santa Monica,0.000692
20145,God of War,positive,War game,0.000724
20146,God of War,positive,game,0.000878
20147,God of War,positive,War series,0.001068
20148,God of War,positive,Santa Monica Studios,0.001286



NEGATIVE PHRASES


,game_name,sentiment,phrase,score


In [46]:
pros_cons_rows = []

for game_name in games:

    result = generate_pros_cons(
        game_name,
        aspect_analysis,
        key_phrases
    )

    for _, row in result[
        'advantages'
    ].iterrows():

        pros_cons_rows.append({

            'game_name': game_name,

            'type': 'advantage',

            'aspect': row['aspect'],

            'mentions': row['mentions'],

            'sentiment_percentage':
                row['positive_pct']
        })

    for _, row in result[
        'disadvantages'
    ].iterrows():

        pros_cons_rows.append({

            'game_name': game_name,

            'type': 'disadvantage',

            'aspect': row['aspect'],

            'mentions': row['mentions'],

            'sentiment_percentage':
                row['negative_pct']
        })

game_pros_cons = pd.DataFrame(
    pros_cons_rows
)

PROS_CONS_FILE = (
    f'{PROJECT_PATH}/game_pros_cons.csv'
)

game_pros_cons.to_csv(
    PROS_CONS_FILE,
    index=False
)

print(
    "✓ Layer 7 complete"
)

print(
    "✓ Saved:",
    PROS_CONS_FILE
)

✓ Layer 7 complete
✓ Saved: /content/drive/MyDrive/Game_Review_NLP/game_pros_cons.csv


Key Review Points

In [47]:
def generate_key_points(
    game_name,
    aspect_data,
    phrase_data
):

    aspects = aspect_data[
        aspect_data['game_name']
        == game_name
    ].copy()

    phrases = phrase_data[
        phrase_data['game_name']
        == game_name
    ].copy()

    points = []

    if aspects.empty:

        return points

    # Most discussed
    most_discussed = aspects.loc[
        aspects['mentions'].idxmax()
    ]

    points.append(
        f"Most discussed aspect: "
        f"{most_discussed['aspect']} "
        f"({most_discussed['mentions']} reviews)"
    )

    # Strongest positive
    strongest = aspects.loc[
        aspects['positive_pct'].idxmax()
    ]

    points.append(
        f"Strongest aspect: "
        f"{strongest['aspect']} "
        f"({strongest['positive_pct']:.1f}% positive)"
    )

    # Most criticized
    criticized = aspects.loc[
        aspects['negative_pct'].idxmax()
    ]

    points.append(
        f"Most criticized aspect: "
        f"{criticized['aspect']} "
        f"({criticized['negative_pct']:.1f}% negative)"
    )

    # Positive phrase
    positive = phrases[
        phrases['sentiment']
        == 'positive'
    ]

    if not positive.empty:

        points.append(
            "Common positive theme: "
            + str(
                positive.iloc[0]['phrase']
            )
        )

    # Negative phrase
    negative = phrases[
        phrases['sentiment']
        == 'negative'
    ]

    if not negative.empty:

        points.append(
            "Common negative theme: "
            + str(
                negative.iloc[0]['phrase']
            )
        )

    return points

In [48]:
key_points = generate_key_points(
    test_game,
    aspect_analysis,
    key_phrases
)

print(
    f"🎮 {test_game}"
)

print("=" * 60)

for point in key_points:

    print(
        "•",
        point
    )

🎮 God of War
• Most discussed aspect: Story (72 reviews)
• Strongest aspect: Graphics (100.0% positive)
• Most criticized aspect: Gameplay (0.0% negative)
• Common positive theme: God of War


In [49]:
key_point_rows = []

for game_name in games:

    points = generate_key_points(
        game_name,
        aspect_analysis,
        key_phrases
    )

    for point in points:

        key_point_rows.append({

            'game_name': game_name,

            'key_point': point
        })

game_key_points = pd.DataFrame(
    key_point_rows
)

KEY_POINTS_FILE = (
    f'{PROJECT_PATH}/game_key_points.csv'
)

game_key_points.to_csv(
    KEY_POINTS_FILE,
    index=False
)

print(
    "✓ Layer 8 complete"
)

print(
    "✓ Saved:",
    KEY_POINTS_FILE
)

✓ Layer 8 complete
✓ Saved: /content/drive/MyDrive/Game_Review_NLP/game_key_points.csv


Game Comparison

In [50]:
game_sentiment = (
    predictions
    .groupby(
        [
            'game_name',
            'predicted_sentiment'
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

for sentiment in [
    'positive',
    'neutral',
    'negative'
]:

    if sentiment not in game_sentiment.columns:

        game_sentiment[
            sentiment
        ] = 0

game_sentiment[
    'total_reviews'
] = game_sentiment[
    [
        'positive',
        'neutral',
        'negative'
    ]
].sum(axis=1)

game_sentiment[
    'positive_pct'
] = (
    game_sentiment['positive']
    /
    game_sentiment['total_reviews']
    * 100
)

game_sentiment[
    'neutral_pct'
] = (
    game_sentiment['neutral']
    /
    game_sentiment['total_reviews']
    * 100
)

game_sentiment[
    'negative_pct'
] = (
    game_sentiment['negative']
    /
    game_sentiment['total_reviews']
    * 100
)

game_sentiment = (
    game_sentiment
    .reset_index()
)

display(
    game_sentiment.head()
)

predicted_sentiment,game_name,negative,neutral,positive,total_reviews,positive_pct,neutral_pct,negative_pct
0,007 First Light,1,5,132,138,95.652174,3.623188,0.724638
1,1000xRESIST,0,2,27,29,93.103448,6.896552,0.000000
2,13 Sentinels Aegis Rim,0,7,112,119,94.117647,5.882353,0.000000
3,2XKO,0,1,9,10,90.000000,10.000000,0.000000
4,3 Out of 10: Season Two,0,2,1,3,33.333333,66.666667,0.000000


comparison score

In [51]:
game_sentiment[
    'sentiment_score'
] = (

    game_sentiment[
        'positive_pct'
    ]

    -

    game_sentiment[
        'negative_pct'
    ]
)

game_comparison = (
    game_sentiment
    .sort_values(
        'sentiment_score',
        ascending=False
    )
)

display(
    game_comparison[
        [
            'game_name',
            'total_reviews',
            'positive_pct',
            'neutral_pct',
            'negative_pct',
            'sentiment_score'
        ]
    ].head(20)
)

predicted_sentiment,game_name,total_reviews,positive_pct,neutral_pct,negative_pct,sentiment_score
13,A Mortician's Tale,5,100.0,0.0,0.0,100.0
1999,​Red Matter,8,100.0,0.0,0.0,100.0
724,Hearthstone - Knights of the Frozen Throne,3,100.0,0.0,0.0,100.0
727,Hearthstone: March of the Lich King,3,100.0,0.0,0.0,100.0
708,HYPER DEMON,8,100.0,0.0,0.0,100.0
729,Hearthstone: The Grand Tournament,3,100.0,0.0,0.0,100.0
730,Hearthstone: The League of Explorers,3,100.0,0.0,0.0,100.0
745,Highway Blossoms,4,100.0,0.0,0.0,100.0
755,Hollow Knight: Voidheart Edition,12,100.0,0.0,0.0,100.0
760,Horizon Chase Turbo: Senna Forever,10,100.0,0.0,0.0,100.0


In [52]:
COMPARISON_FILE = (
    f'{PROJECT_PATH}/game_comparison.csv'
)

game_comparison.to_csv(
    COMPARISON_FILE,
    index=False
)

print(
    "✓ Layer 9 complete"
)

print(
    "✓ Saved:",
    COMPARISON_FILE
)

✓ Layer 9 complete
✓ Saved: /content/drive/MyDrive/Game_Review_NLP/game_comparison.csv
